# ContractNLI v2 — 기본 동작 비교와 출력 검증
Windows · PyCharm Jupyter · **Python 3.11.9 · RTX 5070 12GB**.

- Original, Naive, Contextualized, LongLLMLingua, **3개 정답 클래스 모두 유지**.
- 요약 두 조건: 길이 목표 없음, 최대 **2,048 생성 토큰**. 이는 실험 상한이며 Qwen 공식 기본값이라는 뜻이 아닙니다.
- LongLLMLingua: 공식 README 예제 **rate=0.55**, target_token 덮어쓰기 없음. 함수 자체의 rate 기본값은 0.5입니다.
- Qwen3 판단: thinking off, 세 라벨만 허용하는 제약 생성. 추가 학습은 하지 않습니다.
- 실제 길이는 맞추지 않습니다. 이 실행은 기본 동작 비교이고 동일 길이 실험은 별도 분석입니다.

**v1 폴더와 별도의 새 프로젝트 폴더에 이 버전을 설치하세요.** 기존 outputs와 캐시는 보존합니다.
소스/프로토콜이 바뀌었으므로 v1 체크포인트를 v2 추론에 재사용하지 않습니다. 분석은 마지막 audit 셀로 읽기만 합니다.

## 0. PyCharm 준비
새 ZIP을 기존 폴더와 다른 위치에 풉니다. Python 3.11.9 가상환경과 Jupyter 커널을 같은 것으로 선택하세요.
필요한 경우 PyCharm Terminal에서 `python -m pip install jupyter ipykernel`을 실행합니다.
Git 설치는 필요하지 않습니다. Hugging Face 다운로드 캐시는 같은 Windows 계정에서 자동 재사용됩니다.

## 1. 환경 확인 — 재시작 후에도 실행

In [ ]:
import os
import sys
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path

print("Python:", sys.version)
print("Kernel executable:", sys.executable)
if sys.version_info[:3] != (3, 11, 9):
    raise RuntimeError(
        "이 노트북은 Python 3.11.9 기준입니다. PyCharm에서 해당 가상환경의 커널을 선택하세요."
    )

# 자동 검색에 실패할 때만 수정하세요. 예: r"C:\Users\YourName\PycharmProjects\JJJ"
PROJECT_ROOT_OVERRIDE = ""

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(
        (p for p in candidates
         if (p / "configs/rtx5070.toml").is_file()
         and (p / "src/contractnli").is_dir()),
        None,
    )

if PROJECT_ROOT is None or not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("PROJECT_ROOT_OVERRIDE에 git clone한 JJJ 폴더 경로를 지정하세요.")

os.chdir(PROJECT_ROOT)

# torch를 처음 import하기 전에 GPU를 한 개만 노출합니다.
if "torch" in sys.modules and os.environ.get("CUDA_VISIBLE_DEVICES") != "0":
    raise RuntimeError("커널을 재시작한 뒤 이 셀부터 다시 실행하세요.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Hugging Face를 import하기 전에 다운로드 대기 시간을 설정합니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
print("Project root:", PROJECT_ROOT)
print("Visible physical GPU:", os.environ["CUDA_VISIBLE_DEVICES"])

## 2. 설치 — 새 가상환경에서 한 번
각 설치 셀을 따로 실행하세요. RTX 5070은 torch 2.8.0 / CUDA 12.8 조합을 사용합니다.
설치 후 커널을 재시작하고 **1번 환경 셀 → 3번 GPU 셀**로 이동합니다. Run All로 설치와 추론을 한꺼번에 실행하지 마세요.

In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install --upgrade torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
%pip install -e ".[inference,quantization,longllmlingua,rtx5070,analysis]" ipywidgets

## 3. 커널 재시작 후 GPU 확인
재시작 버튼이 보이지 않으면 코드 셀을 클릭하고 **Ctrl+Shift+A → Restart Kernel**을 검색하세요.
재시작 후 1번 환경 셀을 먼저 다시 실행합니다. 아래 두 실제 연산 검사가 모두 PASS여야 합니다.

In [ ]:
import gc
import torch
import transformers
import bitsandbytes as bnb

GPU_CHECK_PASSED = False
print("PyTorch:", torch.__version__)
print("PyTorch wheel CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA를 사용할 수 없습니다. 커널, CUDA wheel 설치, NVIDIA 드라이버를 확인하세요.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("커널을 재시작하고 1번 셀에서 GPU 0 노출 설정을 적용하세요.")
if torch.__version__.split("+")[0] != "2.8.0" or torch.version.cuda != "12.8":
    raise RuntimeError("이 실험 프로필은 torch 2.8.0/CUDA 12.8입니다. 설치 셀 실행 후 커널을 재시작하세요.")
if bnb.__version__ != "0.48.1":
    raise RuntimeError("rtx5070 extra로 bitsandbytes 0.48.1을 설치하고 커널을 재시작하세요.")

properties = torch.cuda.get_device_properties(0)
capability = torch.cuda.get_device_capability(0)
architectures = torch.cuda.get_arch_list()
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("GPU:", properties.name)
print("Compute capability:", capability)
print("Wheel architectures:", architectures)
print(f"VRAM total: {properties.total_memory / 1024**3:.2f} GiB")
print(f"VRAM currently free: {free_bytes / 1024**3:.2f} GiB")
if capability >= (12, 0) and "sm_120" not in architectures:
    raise RuntimeError("현재 PyTorch wheel에 Blackwell sm_120 지원이 없습니다. CUDA 12.8 설치 셀을 확인하세요.")

# CUDA available=True만으로 실제 커널 호환성을 확인할 수는 없습니다.
with torch.inference_mode():
    matrix = torch.randn(64, 64, device="cuda:0", dtype=torch.float16)
    product = matrix @ matrix.T
    torch.cuda.synchronize()
    if not torch.isfinite(product).all().item():
        raise RuntimeError("CUDA 연산 결과가 유효하지 않습니다.")
print("CUDA tensor test: PASS")

# 대형 모델을 받기 전에 Windows/Blackwell NF4 커널을 직접 확인합니다.
with torch.inference_mode():
    layer = bnb.nn.Linear4bit(
        64, 32, bias=False, compute_dtype=torch.float16,
        quant_type="nf4", compress_statistics=True,
    ).to("cuda:0")
    probe_input = torch.randn(2, 64, device="cuda:0", dtype=torch.float16)
    probe_output = layer(probe_input)
    torch.cuda.synchronize()
    if probe_output.shape != (2, 32) or not torch.isfinite(probe_output).all().item():
        raise RuntimeError("NF4 4bit 연산 결과가 유효하지 않습니다.")
print("bitsandbytes NF4 test: PASS")

del matrix, product, layer, probe_input, probe_output
gc.collect()
torch.cuda.empty_cache()
GPU_CHECK_PASSED = True
print("GPU compatibility checks: PASS. 다음 설정 셀로 진행하세요.")

## 4. 프로토콜·모델 revision 고정
처음에는 메타데이터만 읽어 모델 세 개의 실제 commit을 고정합니다. 가중치를 로딩하지 않습니다.
`configs/rtx5070_v2.frozen.json`은 이후 실행에도 재사용합니다. 모델이나 설정을 바꾸려면 새 config와 출력 폴더를 만드세요.

In [ ]:
if not globals().get("GPU_CHECK_PASSED", False):
    raise RuntimeError("GPU 실제 연산 검사부터 통과하세요.")
import json
from dataclasses import replace
import contractnli
from contractnli.config import load_config
from contractnli.reproducibility import freeze_config
from contractnli.experiment import Experiment, inspect
from contractnli.data import load_dataset
from contractnli.download import download
from contractnli.preflight import preflight

expected_package = PROJECT_ROOT / "src" / "contractnli" / "__init__.py"
if Path(contractnli.__file__).resolve() != expected_package.resolve():
    raise RuntimeError("다른 프로젝트의 패키지가 로딩됐습니다. 이 폴더에서 설치한 뒤 커널을 재시작하세요.")
BASE_CONFIG = freeze_config(load_config(PROJECT_ROOT / "configs" / "rtx5070_v2.toml"),
                            PROJECT_ROOT / "configs" / "rtx5070_v2.frozen.json")
print(json.dumps(BASE_CONFIG.to_dict(), indent=2, ensure_ascii=False))

## 5. 데이터 확인
전체 test는 123문서, 2,091문항입니다. 실행 검사는 dev 1문서를 사용합니다.
이전에 test 결과를 확인했으므로 이후 조정은 탐색적 실험입니다. 프롬프트·모델 설정의 추가 튜닝은 dev에서 수행하세요.

In [ ]:
DATA_PATH = PROJECT_ROOT / BASE_CONFIG.dataset
DEV_PATH = DATA_PATH.with_name("dev.json")
if not DATA_PATH.is_file() or not DEV_PATH.is_file():
    download(DATA_PATH.parent)
from contractnli.data import load_for_config
dataset = load_for_config(BASE_CONFIG)
inventory = inspect(BASE_CONFIG, dataset)
print(json.dumps(inventory, indent=2, ensure_ascii=False))
if inventory["documents"] != 123 or inventory["evaluated_pairs"] != 2091:
    raise RuntimeError("공식 전체 test 데이터가 아닙니다.")
SMOKE_CONFIG = replace(BASE_CONFIG, dataset=str(DEV_PATH), max_documents=1,
                       expected_dataset_sha256="310af7d661d2ab50ee3700169cef524c75f39fb296bbf5a515c229eb0f42e68e",
                       output_dir="outputs/rtx5070_native_v2_dev_smoke")

## 6. 입력 길이 검사
원문을 자동으로 자르지 않습니다. 요약은 입력 길이 + 2,048 출력 토큰을 확인합니다.
이 단계는 토크나이저만 사용하므로 VRAM 적재 여부는 다음 단계에서 확인합니다.
LongLLMLingua prefix 호환 처리는 코드에 통합되어 별도 패치 셀이 필요하지 않습니다.

In [ ]:
length_check = preflight(BASE_CONFIG)
print(json.dumps(length_check, indent=2, ensure_ascii=False))
if not length_check["ok"]:
    raise RuntimeError("입력 길이 한도를 초과했습니다. 원문을 임의로 자르지 마세요.")

## 7. dev 한 문서 실행 검사
68개 판단을 완료할 때까지 기다리세요. 첫 실행은 모델 다운로드도 필요합니다.
모델마다 고정된 합성 예제로 1회 워밍업하며, 워밍업은 점수와 처리시간 집계에서 제외됩니다.
`Loading`, `summary`, `compression`, `judgment` 로그를 구분해서 보세요.
**Smoke run completed: True**와 Invalid=0을 확인한 뒤 8번으로 이동합니다.

In [ ]:
smoke_report = None
print("dev 한 문서 실행 검사를 시작합니다. 완료될 때까지 중지하지 마세요.", flush=True)
smoke_report = Experiment(SMOKE_CONFIG).run("all")
for condition, m in smoke_report["metrics"].items():
    print(condition, "Invalid:", m["invalid_outputs"], "판단 길이 제한 도달:", m["judgment_length_capped_calls"])
    if m["invalid_outputs"] or m["judgment_length_capped_calls"]:
        raise RuntimeError("출력 프로토콜 검증 실패. 전체 실행을 시작하지 마세요.")
print("Smoke run completed:", smoke_report["complete"])
print("실행 검사 결과는 dev 부분집합으로 test 성능이 아닙니다.")

## 8. 전체 메인 실험
7번이 끝나면 아래 변수만 True로 바꾸고 이 셀을 실행합니다.
단일 GPU에서 요약 → 압축 → 판단 모델을 순서대로 올리고 해제합니다.
판단은 특정 조건이 항상 먼저 실행되지 않도록 seed=42의 고정 해시 순서로 섞습니다.
일반 중단 후 같은 셀을 재실행하면 완료된 항목을 재사용합니다. 기존 v1 결과는 읽거나 덮어쓰지 않습니다.

In [ ]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    if not isinstance(globals().get("smoke_report"), dict) or not smoke_report.get("complete"):
        raise RuntimeError("7번 dev 실행 검사가 완료되어야 합니다.")
    if any(m["invalid_outputs"] for m in smoke_report["metrics"].values()):
        raise RuntimeError("dev 출력 검증을 통과하지 못했습니다.")
    full_report = Experiment(BASE_CONFIG).run("all")
    print("전체 실험 완료:", full_report["complete"])
    print("결과 폴더:", (PROJECT_ROOT / BASE_CONFIG.output_dir).resolve())
else:
    print("전체 실행은 꺼져 있습니다. dev 검사가 끝난 뒤 True로 바꾸세요.")

## 9. 성능·형식·시간 함께 확인
시간 단위는 분입니다. 총시간은 요약/압축 + 판단이며, 모델 준비·워밍업·체크포인트 I/O는 제외합니다.
모델 준비와 실행 전체 경과시간은 run.json에 별도로 기록합니다. 저장된 시간을 재사용한 실행은 새 벤치마크가 아닙니다.

In [ ]:
from IPython.display import Markdown, display
result_config = BASE_CONFIG if globals().get("RUN_FULL_EXPERIMENT", False) else SMOKE_CONFIG
result_directory = PROJECT_ROOT / result_config.output_dir
metrics_path = result_directory / "metrics.json"
if not metrics_path.exists():
    print("아직 전체 예측이 완료되지 않았습니다:", result_directory)
else:
    report = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("부분집합:", report["dataset"]["partial_dataset"], "문항 수:", report["dataset"]["evaluated_pairs"])
    table = ["| 조건 | Accuracy | Macro-F1 | Invalid | 실제 보존율 | 원문 정답 유지율 |",
             "|---|---:|---:|---:|---:|---:|"]
    timing = ["| 조건 | 요약·압축(분) | 판단(분) | 합계(분) | 문항당(초) | 판단 p95(초) |",
              "|---|---:|---:|---:|---:|---:|"]
    for condition, m in report["metrics"].items():
        retention = m.get("correct_decision_retention")
        keep = "—" if retention is None else f"{retention:.2%}"
        table.append(f"| {condition} | {m['accuracy']:.2%} | {m['macro_f1']:.4f} | {m['invalid_outputs']} | {m['mean_actual_token_retention']:.2%} | {keep} |")
        timing.append(f"| {condition} | {m['compression_seconds']/60:.2f} | {m['judgment_seconds']/60:.2f} | {m['total_inference_seconds']/60:.2f} | {m['amortized_seconds_per_pair']:.3f} | {m['judgment_p95_seconds']:.3f} |")
        print(condition, "요약 생성 상한 도달:", m["length_capped_summary_calls"], "판단 출력 상한 도달:", m["judgment_length_capped_calls"])
    display(Markdown("\n".join(table)))
    display(Markdown("\n".join(timing)))
    print("JSON:", metrics_path)
    print("Excel용 CSV:", result_directory / "metrics.csv")

## 10. 선택: 기존 또는 새 결과의 읽기 전용 분석
모델을 올리지 않습니다. Accuracy/Macro-F1 재계산, 엄격한 파싱/첫 줄 파싱 비교, 클래스별·가설별 지표,
원문 정답 보존, 문서 단위 bootstrap 20,000회를 별도 audit 폴더에 저장합니다.

길이 겹침 진단은 같은 문항에서 보존율 차이가 2%p 이내인 부분집합만 보여줍니다. 선택된 표본이므로
동일 길이 전체 실험으로 해석하면 안 됩니다. 정확히 같은 토큰 수에서 비용/시간을 비교하는 벤치마크는 별도 프로토콜입니다.

In [ ]:
RUN_AUDIT = False
# 기존 v1 분석 시 이 경로만 기존 결과 폴더로 바꾸세요. 원본 파일은 변경하지 않습니다.
AUDIT_SOURCE = PROJECT_ROOT / BASE_CONFIG.output_dir
if RUN_AUDIT:
    from contractnli.audit import save_audit
    audit = save_audit(AUDIT_SOURCE / "predictions.jsonl", AUDIT_SOURCE / "metrics.json",
                       PROJECT_ROOT / "outputs" / "native_v2_audit", bootstrap_samples=20000)
    print("분석 완료:", PROJECT_ROOT / "outputs" / "native_v2_audit")

## 해석과 재현 범위
- 메인 결과는 실제 길이가 다른 기본 동작 비교입니다. 길이 효과와 방법 효과를 분리해 주장하지 않습니다.
- Original의 오답 유지와 정답 유지는 다릅니다. 정답 유지율·정답→오답·오답→정답을 함께 보세요.
- NotMentioned는 정답 라벨로 유지하되, 없는 gold evidence를 근거 보존 0점으로 계산하지 않습니다.
- 생성 상한에 도달한 요약은 잘린 문장인지 확인합니다. 자동 연장·절단·정답 기반 재시도는 하지 않습니다.
- Constrained decoding은 출력 형식을 보장합니다. 정답률·편향 제거·비트 단위 완전 재현을 보장하는 것은 아닙니다.
- 제약 생성과 길이 설정이 바뀌었으므로 v1과 v2는 별도 프로토콜입니다. 기존 결과를 새 결과로 덮어쓰지 않습니다.
- 모델 변경이나 추가 학습을 진행하면 별도 실험으로 기록하고 train/dev만 학습·튜닝에 사용하세요.